In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import time
import math
import copy
import matplotlib.pyplot as plt

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"device: {device}")

# load + time split (same cutoff as before)
ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df_full = ratings[ratings["timestamp"] < cutoff_ts].reset_index(drop=True)
val_df_full   = ratings[ratings["timestamp"] >= cutoff_ts].reset_index(drop=True)

# build mappings from train (any rating)
user_to_idx = {u: i for i, u in enumerate(train_df_full["userId"].unique())}
movie_to_idx = {m: i for i, m in enumerate(train_df_full["movieId"].unique())}
n_users = len(user_to_idx)
n_movies = len(movie_to_idx)

def apply_mappings(df):
    df = df.copy()
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    df = df.dropna(subset=["user_idx", "movie_idx"]).reset_index(drop=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)
    return df

train_df_full = apply_mappings(train_df_full)
val_df_full   = apply_mappings(val_df_full)

LIKE_THRESHOLD = 4.0
train_pos = train_df_full[train_df_full["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)
val_pos   = val_df_full[val_df_full["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)

print(f"users: {n_users:,} | movies: {n_movies:,}")
print(f"\ntrain (any rating): {len(train_df_full):,}")
print(f"train positives (>= {LIKE_THRESHOLD}): {len(train_pos):,} ({len(train_pos)/len(train_df_full)*100:.1f}%)")
print(f"\nval (any rating): {len(val_df_full):,}")
print(f"val positives (>= {LIKE_THRESHOLD}): {len(val_pos):,} ({len(val_pos)/len(val_df_full)*100:.1f}%)")

device: mps
users: 150,330 | movies: 45,058

train (any rating): 22,454,535
train positives (>= 4.0): 11,177,787 (49.8%)

val (any rating): 454,141
val positives (>= 4.0): 181,949 (40.1%)


In [2]:
class BPRDataset(Dataset):
    """
    yields (user_idx, pos_movie_idx, neg_movie_idx) triples.
    
    positives come from `pos_df` (rating >= threshold).
    negatives are sampled uniformly at random from movies the user has NOT rated
    (we use the full train ratings df, not just positives, to avoid sampling 
    a movie the user rated 1.0 — that's not a "negative", it's a known dislike).
    """
    def __init__(self, pos_df: pd.DataFrame, all_train_df: pd.DataFrame, n_movies: int, seed: int = 0):
        # positives as torch tensors
        self.users = torch.from_numpy(pos_df["user_idx"].values.astype(np.int64))
        self.pos_movies = torch.from_numpy(pos_df["movie_idx"].values.astype(np.int64))
        
        # for each user, the set of movies they've already interacted with (any rating)
        # we won't sample these as negatives
        self.user_to_seen = {}
        for user_idx, group in all_train_df.groupby("user_idx"):
            self.user_to_seen[int(user_idx)] = set(group["movie_idx"].values.tolist())
        
        self.n_movies = n_movies
        self.rng = np.random.RandomState(seed)
    
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, i):
        u = int(self.users[i].item())
        pos_m = int(self.pos_movies[i].item())
        seen = self.user_to_seen[u]
        
        # rejection sampling: draw a random movie until we get one not in seen
        while True:
            neg_m = self.rng.randint(0, self.n_movies)
            if neg_m not in seen:
                break
        return self.users[i], self.pos_movies[i], torch.tensor(neg_m, dtype=torch.long)


# build datasets
print("building train dataset (this builds the user_to_seen lookup)...")
t0 = time.time()
train_ds = BPRDataset(train_pos, train_df_full, n_movies, seed=42)
print(f"train dataset built in {time.time()-t0:.1f}s, {len(train_ds):,} positive interactions")

# quick sanity check
u, p, n = train_ds[0]
print(f"\nsample 0: user={u.item()}, pos_movie={p.item()}, neg_movie={n.item()}")

# verify negative isn't in user's seen set
seen_for_u = train_ds.user_to_seen[int(u.item())]
print(f"pos_movie in seen set? {p.item() in seen_for_u}")
print(f"neg_movie in seen set? {n.item() in seen_for_u}")

building train dataset (this builds the user_to_seen lookup)...
train dataset built in 4.2s, 11,177,787 positive interactions

sample 0: user=0, pos_movie=2, neg_movie=15795
pos_movie in seen set? True
neg_movie in seen set? False


In [3]:
class TwoTower(nn.Module):
    """
    user tower: userId -> user vector (dim)
    item tower: movieId -> item vector (dim)
    score(u, m) = user_emb(u) · item_emb(m)
    
    for retrieval at serving time: precompute all item vectors, then for any 
    user, do user_emb(u) once and find nearest items via ann (faiss).
    
    no biases here — bpr loss only cares about score *gaps*, so a bias term 
    cancels out in score(u, m+) - score(u, m-). adding biases would just 
    add capacity without changing the loss landscape.
    """
    def __init__(self, n_users: int, n_movies: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
        # init: small std so initial dot products are tiny, training has room to grow magnitudes
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
    
    def encode_user(self, u: torch.Tensor) -> torch.Tensor:
        return self.user_emb(u)
    
    def encode_item(self, m: torch.Tensor) -> torch.Tensor:
        return self.item_emb(m)
    
    def score(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        """elementwise dot product per row."""
        return (user_vec * item_vec).sum(dim=-1)
    
    def forward(self, u: torch.Tensor, m_pos: torch.Tensor, m_neg: torch.Tensor):
        """returns (pos_score, neg_score) for bpr loss."""
        uv = self.encode_user(u)
        pv = self.encode_item(m_pos)
        nv = self.encode_item(m_neg)
        return self.score(uv, pv), self.score(uv, nv)


EMB_DIM = 64  # bigger than mf's 32; ranking benefits from more capacity
model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\ntotal params: {n_params:,}")
print(f"  user_emb:  {n_users * EMB_DIM:,}  ({n_users:,} users × {EMB_DIM} dim)")
print(f"  item_emb:  {n_movies * EMB_DIM:,}  ({n_movies:,} movies × {EMB_DIM} dim)")

# test forward pass
u, p, n = next(iter(DataLoader(train_ds, batch_size=64)))
u, p, n = u.to(device), p.to(device), n.to(device)
with torch.no_grad():
    pos_score, neg_score = model(u, p, n)
print(f"\ntest forward:")
print(f"  pos_score range: [{pos_score.min().item():+.4f}, {pos_score.max().item():+.4f}]")
print(f"  neg_score range: [{neg_score.min().item():+.4f}, {neg_score.max().item():+.4f}]")
print(f"  pos > neg in {(pos_score > neg_score).float().mean().item()*100:.1f}% of pairs")

TwoTower(
  (user_emb): Embedding(150330, 64)
  (item_emb): Embedding(45058, 64)
)

total params: 12,504,832
  user_emb:  9,621,120  (150,330 users × 64 dim)
  item_emb:  2,883,712  (45,058 movies × 64 dim)

test forward:
  pos_score range: [-0.0024, +0.0020]
  neg_score range: [-0.0025, +0.0021]
  pos > neg in 53.1% of pairs


In [4]:
BATCH_SIZE = 8192
LR = 0.005
WEIGHT_DECAY = 1e-5
N_EPOCHS = 3

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"train batches per epoch: {len(train_loader):,}")

# bpr loss: -log(sigmoid(pos_score - neg_score))
# we use -F.logsigmoid (numerically stable) instead of -torch.log(sigmoid(...))
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

losses = []
pos_gt_neg = []  # fraction of training pairs where pos_score > neg_score

import torch.nn.functional as F

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_seen = 0
    t0 = time.time()
    
    for u, p, n in train_loader:
        u, p, n = u.to(device), p.to(device), n.to(device)
        
        optimizer.zero_grad()
        pos_score, neg_score = model(u, p, n)
        
        # bpr loss
        loss = -F.logsigmoid(pos_score - neg_score).mean()
        
        loss.backward()
        optimizer.step()
        
        bsz = len(u)
        epoch_loss += loss.item() * bsz
        epoch_correct += (pos_score > neg_score).sum().item()
        n_seen += bsz
    
    avg_loss = epoch_loss / n_seen
    accuracy = epoch_correct / n_seen
    losses.append(avg_loss)
    pos_gt_neg.append(accuracy)
    
    print(f"epoch {epoch}/{N_EPOCHS} | bpr loss: {avg_loss:.4f} | "
          f"pos>neg: {accuracy*100:.1f}% | {time.time()-t0:.1f}s")

print("\ndone")

train batches per epoch: 1,365
epoch 1/3 | bpr loss: 0.2918 | pos>neg: 95.1% | 230.4s
epoch 2/3 | bpr loss: 0.2634 | pos>neg: 97.0% | 270.3s
epoch 3/3 | bpr loss: 0.2623 | pos>neg: 97.1% | 264.2s

done


In [5]:
def _dcg(rels):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))


def evaluate_ranking_two_tower(model, train_df, val_df, n_movies, k_list=(5, 10, 20),
                               n_sample_users=1000, like_threshold=4.0, seed=42):
    rng = np.random.RandomState(seed)
    
    train_by_user = train_df.groupby("user_idx")["movie_idx"].apply(set)
    val_by_user_liked = (
        val_df[val_df["rating"] >= like_threshold]
        .groupby("user_idx")["movie_idx"].apply(set)
    )
    eligible = list(val_by_user_liked.index)
    print(f"eligible val users: {len(eligible):,}")
    sample_users = rng.choice(eligible, size=min(n_sample_users, len(eligible)), replace=False)
    
    pop_score = np.zeros(n_movies, dtype=np.float32)
    counts = train_df["movie_idx"].value_counts()
    pop_score[counts.index.values] = counts.values
    
    model.eval()
    
    # precompute all item embeddings once — this is the two-tower advantage
    with torch.no_grad():
        all_movies_t = torch.arange(n_movies, dtype=torch.long, device=device)
        item_vecs = model.encode_item(all_movies_t)  # (n_movies, dim)
    print(f"precomputed all {n_movies:,} item vectors, shape {tuple(item_vecs.shape)}")
    
    metrics = {f"recall@{k}": [] for k in k_list}
    metrics.update({f"ndcg@{k}": [] for k in k_list})
    metrics.update({f"pop_recall@{k}": [] for k in k_list})
    
    t0 = time.time()
    with torch.no_grad():
        for user_idx in sample_users:
            seen = train_by_user.get(user_idx, set())
            liked = val_by_user_liked[user_idx]
            
            mask = np.ones(n_movies, dtype=bool)
            mask[list(seen)] = False
            
            # encode user once, dot with all items
            user_t = torch.tensor([int(user_idx)], dtype=torch.long, device=device)
            user_vec = model.encode_user(user_t)  # (1, dim)
            scores = (item_vecs @ user_vec.T).squeeze(1).cpu().numpy()  # (n_movies,)
            scores[~mask] = -np.inf
            
            pop_masked = pop_score.copy()
            pop_masked[~mask] = -np.inf
            
            for k in k_list:
                top_model = np.argpartition(-scores, k)[:k]
                hits_model = liked.intersection(top_model.tolist())
                metrics[f"recall@{k}"].append(len(hits_model) / len(liked))
                
                top_sorted = top_model[np.argsort(-scores[top_model])]
                rels = [1 if m in liked else 0 for m in top_sorted]
                ideal = [1] * min(k, len(liked))
                ndcg = _dcg(rels) / _dcg(ideal) if ideal else 0
                metrics[f"ndcg@{k}"].append(ndcg)
                
                top_pop = np.argpartition(-pop_masked, k)[:k]
                hits_pop = liked.intersection(top_pop.tolist())
                metrics[f"pop_recall@{k}"].append(len(hits_pop) / len(liked))
    print(f"eval done in {time.time()-t0:.1f}s")
    return {key: float(np.mean(vals)) for key, vals in metrics.items()}


print("evaluating two-tower on 1000 val users...")
results = evaluate_ranking_two_tower(model, train_df_full, val_df_full, n_movies)

print(f"\n{'metric':<18s} {'two-tower':>10s} {'popularity':>12s} {'mf (day 9)':>12s} {'delta vs pop':>14s}")
print("-" * 70)
mf_baseline = {"recall@5": 0.0201, "recall@10": 0.0345, "recall@20": 0.0520}
for k in (5, 10, 20):
    tt = results[f"recall@{k}"]
    pop = results[f"pop_recall@{k}"]
    mf = mf_baseline[f"recall@{k}"]
    print(f"{'recall@'+str(k):<18s} {tt:>10.4f} {pop:>12.4f} {mf:>12.4f} {tt-pop:>+14.4f}")
print()
for k in (5, 10, 20):
    print(f"{'ndcg@'+str(k):<18s} {results[f'ndcg@'+str(k)]:>10.4f}")

evaluating two-tower on 1000 val users...
eligible val users: 6,282
precomputed all 45,058 item vectors, shape (45058, 64)
eval done in 2.9s

metric              two-tower   popularity   mf (day 9)   delta vs pop
----------------------------------------------------------------------
recall@5               0.0221       0.0202       0.0201        +0.0019
recall@10              0.0321       0.0301       0.0345        +0.0020
recall@20              0.0520       0.0468       0.0520        +0.0051

ndcg@5                 0.0885
ndcg@10                0.0797
ndcg@20                0.0770


two-tower with bpr loss + uniform random negatives. ndcg@5 0.0885 vs mf 0.0783 (+13%) — real lift in fine-grained ordering. but recall@10 0.0321 vs mf 0.0345 is worse. uniform negatives mostly sample long-tail items, so the model learns "popular vs obscure", not "this user's taste". next: popularity-weighted negative sampling and/or in-batch negatives to get harder negatives.